# 2. Evaluation

For this last future-work phase we will start (but possibly not end it :)


In [3]:
from typing import Dict, List, Tuple, Optional, Any
import re
import json
import time
import logging
from dataclasses import dataclass, field

import torch
import pandas as pd
from tqdm.auto import tqdm
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    GenerationConfig,
)
!pip install loguru
from loguru import logger

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 2.0 MB/s eta 0:00:00


In [9]:
# ---------------------------------------------------------------------
# PedanticJudge
# ---------------------------------------------------------------------
class PedanticJudge:
    def __init__(
        self,
        judge_model_name: str = "Qwen/Qwen2.5-0.5B-Instruct",
        device: Optional[str] = None,
        max_attempts: int = 3,
    ):
        """Initialize the LLM judge model"""
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        logger.info(f"🚀 Loading judge model: {judge_model_name} on {self.device}")

        # Load tokenizer & model (keep default dtype to be conservative)
        self.tokenizer = AutoTokenizer.from_pretrained(judge_model_name, trust_remote_code=True)
        self.model = AutoModelForCausalLM.from_pretrained(
            judge_model_name,
            trust_remote_code=True,
            device_map="auto" if ("cuda" in self.device and torch.cuda.is_available()) else None,
        )

        # ensure pad token
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        # Default generation config for judge calls
        self.generation_config = GenerationConfig(
            do_sample=True,
            temperature=0.3,  # Low temperature for consistent judging
            max_new_tokens=1024,
            pad_token_id=self.tokenizer.eos_token_id,
            eos_token_id=self.tokenizer.eos_token_id,
        )

        # Evaluation criteria
        self.criteria = {
            "vocabulary": "Sophistication and precision of word choice, use of technical terms, avoidance of colloquialisms",
            "pedanticity": "Formality, attention to detail, qualification of statements, academic tone",
            "precision": "Elimination of vague language, use of specific references and concrete details",
            "structure": "Logical organization, clear transitions, systematic presentation",
        }

        # Retry behavior if parsing fails
        self.max_attempts = max_attempts

    def _build_all_criteria_prompt(self, prompt: str, response: str) -> str:
        """
        Ask the judge to output a compact JSON with each criterion having 'score' and 'rationale'.
        Keep instructions explicit and show an example JSON format to increase parseability.
        """
        criteria_block = "\n".join([f"- {k}: {v}" for k, v in self.criteria.items()])
        instruction = f"""
          You are an expert linguistic evaluator. For the following response, evaluate the text on these criteria:
          {criteria_block}

          Return a JSON object ONLY with keys for each criterion. Each criterion value must be an object with:
            - score: a number between 1 and 10 (float allowed)
            - rationale: a short explanation (1-2 sentences)

          Do NOT output anything else outside the JSON. Example:
          {{"vocabulary": {{"score": 7.5, "rationale": "Uses technical terms but occasionally vague."}}, "pedanticity": ...}}

          ORIGINAL PROMPT:
          {prompt}

          RESPONSE TO EVALUATE:
          {response}

          Return JSON now:
        """
        return instruction.strip()

    # This is regex based extraction, lets pray to the gods honestly because small LLMs are kind of rebel and they wont probably output a coherent json format :(
    def _extract_scores_fallback(self, text: str) -> Dict[str, Dict[str, Any]]:
        """
        Fallback extractor: attempts to retrieve numeric scores per criterion via regex.
        Returns structure: {criterion: {"score": float, "rationale": "..."}}
        """
        results = {}
        # try to find patterns like 'vocabulary: 8' or 'Vocabulary - 8/10' or 'Vocabulary: 8 - rationale...'
        for crit in self.criteria.keys():
            pat = re.compile(
                rf"{crit}[:\s\-]*\s*(\d+(?:\.\d+)?)", re.IGNORECASE
            )
            m = pat.search(text)
            if m:
                try:
                    score = float(m.group(1))
                    score = float(max(1.0, min(10.0, score)))
                except Exception:
                    score = 5.0
            else:
                score = 5.0
            # naive rationale extraction: take up to 120 chars after the match or fallback slice
            rationale = ""
            if m:
                start = m.end()
                rationale = text[start : start + 200].strip().split("\n")[0]
            if not rationale:
                # fallback: brief excerpt around criterion
                snippet = text[:200].replace("\n", " ")
                rationale = snippet[:140].strip()
            results[crit] = {"score": score, "rationale": rationale}
        return results

    def _parse_as_json(self, text: str) -> Optional[Dict[str, Dict[str, Any]]]:
        """
        Try to parse model output as JSON. Clean up typical problems if present.
        """
        # Try to find JSON substring
        text = text.strip()
        # heuristics: find first "{" and last "}" and attempt json.loads
        try:
            start = text.find("{")
            end = text.rfind("}")
            if start != -1 and end != -1 and end > start:
                candidate = text[start : end + 1]
                parsed = json.loads(candidate)
                # Validate & normalize
                normalized = {}
                for crit in self.criteria.keys():
                    if crit in parsed:
                        entry = parsed[crit]
                        if isinstance(entry, dict) and "score" in entry:
                            score = float(entry["score"])
                            score = max(1.0, min(10.0, score))
                            rationale = str(entry.get("rationale", "")).strip()
                            normalized[crit] = {"score": score, "rationale": rationale}
                        else:
                            # if direct number
                            try:
                                score = float(entry)
                                normalized[crit] = {"score": max(1.0, min(10.0, score)), "rationale": ""}
                            except Exception:
                                normalized[crit] = {"score": 5.0, "rationale": ""}
                    else:
                        normalized[crit] = {"score": 5.0, "rationale": ""}
                return normalized
        except Exception:
            return None
        return None

    def judge_all_criteria(self, prompt: str, response: str, attempt_delay: float = 0.5) -> Dict[str, Dict[str, Any]]:
        """
        Ask the judge model to provide scores+rationales for ALL criteria in a single JSON.
        Retry a few times if parsing fails.
        """
        system_prompt = self._build_all_criteria_prompt(prompt, response)
        inputs = self.tokenizer(system_prompt, return_tensors="pt", truncation=True, max_length=1024)

        for attempt in range(1, self.max_attempts + 1):
            try:
                with torch.no_grad():
                    # Move tensors to model device
                    inputs = {k: v.to(self.model.device) for k, v in inputs.items()}
                    outputs = self.model.generate(
                        **inputs,
                        generation_config=self.generation_config,
                        max_new_tokens=self.generation_config.max_new_tokens,
                        do_sample=True,
                    )
                raw = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
                # Remove the prompt if returned in full
                raw_suffix = raw
                if system_prompt in raw:
                    raw_suffix = raw.split(system_prompt, 1)[-1].strip()

                # 1) try JSON parse
                parsed = self._parse_as_json(raw_suffix)
                if parsed is not None:
                    return parsed

                # 2) fallback regex extraction
                fallback = self._extract_scores_fallback(raw_suffix)
                # If fallback seems reasonable (not all neutral), accept
                scores = [v["score"] for v in fallback.values()]
                if any(s != 5.0 for s in scores) or attempt == self.max_attempts:
                    return fallback

                # else retry
                logger.debug("Judge output ambiguous, retrying...")
                time.sleep(attempt_delay * attempt)
            except Exception as e:
                logger.warning(f"Judge generation failed on attempt {attempt}: {e}")
                time.sleep(attempt_delay * attempt)

        # Last resort: neutral scores
        return {c: {"score": 5.0, "rationale": ""} for c in self.criteria.keys()}

    def comprehensive_evaluation(self, prompt: str, response: str) -> Dict[str, Any]:
        """Evaluate a response on all criteria (single call)"""
        logger.info(f"🔍 Evaluating prompt (len={len(prompt)}): {prompt[:60]}...")
        parsed = self.judge_all_criteria(prompt, response)
        overall = sum(v["score"] for v in parsed.values()) / len(parsed)
        parsed["overall_score"] = overall
        return parsed


# ---------------------------------------------------------------------
# ModelBenchmark
# ---------------------------------------------------------------------
@dataclass
class ModelBenchmark:
    judge: PedanticJudge
    results: List[Dict[str, Any]] = field(default_factory=list)

    def load_model(self, model_name: str):
        """Load a model+tokenizer from HF hub (returns tokenizer, model)"""
        logger.info(f"📥 Loading model: {model_name}")
        tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            trust_remote_code=True,
            device_map="auto" if torch.cuda.is_available() else None,
        )
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        return model, tokenizer

    def generate_response(self, model, tokenizer, prompt: str, max_new_tokens: int = 256, temperature: float = 0.7) -> str:
        """Generate a single response safely (moves inputs to model device)"""
        model_device = next(model.parameters()).device if any(p.numel() for p in model.parameters()) else torch.device("cpu")
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
        inputs = {k: v.to(model_device) for k, v in inputs.items()}

        gen_cfg = GenerationConfig(
            do_sample=True,
            temperature=temperature,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

        with torch.no_grad():
            outputs = model.generate(**inputs, generation_config=gen_cfg)

        raw = tokenizer.decode(outputs[0], skip_special_tokens=True)
        # try to remove echo of prompt if present
        if raw.startswith(prompt):
            raw = raw[len(prompt) :].strip()
        return raw.strip()

    def run_benchmark(self, models_config: Dict[str, Dict[str, Any]], test_prompts: List[str], num_prompts: int = 5, max_new_tokens: int = 256):
        """
        Run benchmark:
         - models_config: {name: {"model": model_obj, "tokenizer": tokenizer_obj, "type": label}}
        """
        logger.info("🏁 Starting LLM-as-Judge Benchmark")
        logger.info("=" * 60)
        models = models_config.copy()

        # Use first N prompts
        test_subset = test_prompts[:num_prompts]

        for i, prompt in enumerate(tqdm(test_subset, desc="Prompts"), start=1):
            logger.info(f"\n📊 Test {i}/{len(test_subset)}: {prompt}")
            for model_name, model_info in models.items():
                label = model_info.get("type", model_name)
                logger.info(f"\n🤖 Generating with {label} ({model_name}):")
                try:
                    response = self.generate_response(
                        model_info["model"],
                        model_info["tokenizer"],
                        prompt,
                        max_new_tokens=max_new_tokens,
                    )
                except Exception as e:
                    logger.error(f"Generation failed for {model_name}: {e}")
                    response = "[generation_error]"

                logger.info(f"   Response (first 200 chars): {response[:200]}")

                # Judge once for all criteria
                evaluation = self.judge.comprehensive_evaluation(prompt, response)

                # Record per-criterion details and overall
                record = {
                    "prompt": prompt,
                    "model_name": model_name,
                    "model_type": label,
                    "response": response,
                    "overall_score": evaluation["overall_score"],
                }
                # add each criterion
                for crit, v in self.judge.criteria.items():
                    rec_key_score = f"{crit}_score"
                    rec_key_rationale = f"{crit}_rationale"
                    record[rec_key_score] = evaluation[crit]["score"]
                    record[rec_key_rationale] = evaluation[crit]["rationale"]
                self.results.append(record)

        logger.info("🏁 Benchmark complete.")
        return self.results

    def get_results_dataframe(self) -> pd.DataFrame:
        """Return results as pandas DataFrame"""
        return pd.DataFrame(self.results)

    def save_results(self, path_csv: str = "benchmark_results.csv", path_json: str = "benchmark_results.json"):
        """Save results to CSV and JSON"""
        df = self.get_results_dataframe()
        df.to_csv(path_csv, index=False)
        with open(path_json, "w", encoding="utf-8") as f:
            json.dump(self.results, f, ensure_ascii=False, indent=2)
        logger.info(f"Saved results -> {path_csv}, {path_json}")

    def print_summary_statistics(self):
        """Print simple summary statistics"""
        if not self.results:
            logger.warning("No results to analyze")
            return
        df = self.get_results_dataframe()
        summary = (
            df.groupby("model_type")["overall_score"]
            .agg(["mean", "std", "min", "max"])
            .round(2)
            .sort_values("mean", ascending=False)
        )
        logger.info("\n" + "=" * 60)
        logger.info("📈 BENCHMARK SUMMARY STATISTICS")
        logger.info("\n" + summary.to_string())
        best = summary["mean"].idxmax()
        best_score = summary["mean"].max()
        logger.info(f"\n🏆 WINNER: {best} (Average Score: {best_score:.2f}/10)")


In [ ]:
### ANOTHER CELL

test_prompts = [
    "Explain what artificial intelligence is.",
    "What is a convolution in computer vision?",
    "What is a token in natural language processing?",
    "Describe the transformer architecture in machine learning.",
    "What is an embedding?",
    "Explain the difference between supervised and unsupervised learning.",

    "Describe how to cook a chicken.",
    "How do i make cookies?",
    "Which are uncommon typical italian dishes?",
    "Tell me about regional dishes from Veneto, Italy",

    "Explain the biomechanical principles of proper tongue posture.",
    "What is the evidence for dietary influences on craniofacial development?",
    "Describe the relationship between nasal breathing and maxillary growth.",
    "What are the physiological mechanisms behind mewing?",

    "Explain the biomechanics of a handstand push-up.",
    "What are the physiological prerequisites for performing a muscle-up?",
    "Describe the motor learning progression for planche mastery.",
    "What distinguishes concentric from eccentric muscle contractions?",
    "Explain the role of leverage in bodyweight exercise difficulty.",

    "List the most protein rich foods",
    "What is your opinion on paleolithic diet from a scientific perspective?",
    "Explain the physiological effects of intermittent fasting.",
]

repo_names = {
    "grpo_model": "eZWALT/SmolLM2-135M-Pedantic-GRPO",
    "ppo_model": "eZWALT/SmolLM2-135M-Pedantic-PPO",
    "dpo_model": "eZWALT/SmolLM2-135M-Pedantic-DPO",
    "sft_model": "eZWALT/SmolLM2-135M-Pedantic-SFT-Instruct",
    "base_model": "HuggingFaceTB/SmolLM2-135M-Instruct",

}

# Instantiate judge and benchmark
judge = PedanticJudge("Qwen/Qwen2.5-0.5B-Instruct")
benchmark = ModelBenchmark(judge=judge)

# Load models (you can replace with local paths or hub ids)
models_config = {}
for key, repo in repo_names.items():
    try:
        model, tokenizer = benchmark.load_model(repo)
        models_config[key] = {"model": model, "tokenizer": tokenizer, "type": key.upper()}
    except Exception as e:
        logger.error(f"Failed to load {repo}: {e}")

# Run benchmark (first 3 prompts)
results = benchmark.run_benchmark(models_config, test_prompts, num_prompts=3, max_new_tokens=1024)
benchmark.save_results()


2025-10-31 11:53:21.739 | INFO     | __main__:__init__:13 - 🚀 Loading judge model: Qwen/Qwen2.5-0.5B-Instruct on cpu
2025-10-31 11:53:25.960 | INFO     | __main__:load_model:210 - 📥 Loading model: eZWALT/SmolLM2-135M-Pedantic-GRPO


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

2025-10-31 11:54:15.628 | INFO     | __main__:run_benchmark:249 - 🏁 Starting LLM-as-Judge Benchmark
2025-10-31 11:54:15.629 | INFO     | __main__:run_benchmark:250 - ============================================================


Prompts:   0%|          | 0/3 [00:00<?, ?it/s]

2025-10-31 11:54:15.643 | INFO     | __main__:run_benchmark:257 - 
📊 Test 1/3: Explain what artificial intelligence is.
2025-10-31 11:54:15.645 | INFO     | __main__:run_benchmark:260 - 
🤖 Generating with GRPO_MODEL (grpo_model):
`generation_config` default values have been modified to match model-specific defaults: {'bos_token_id': 1}. If this is not desired, please set these values explicitly.


KeyboardInterrupt: 

In [ ]:
benchmark.print_summary_statistics()

## Data Analysis on the Results

In [11]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

IN = Path("../resources/pedanticity_benchmark_llm_and_human.json")
OUT_DIR = Path("../resources")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PDF = OUT_DIR / "model_boxplots.pdf"
OUT_CSV = OUT_DIR / "model_summary.csv"

data = json.loads(IN.read_text(encoding="utf-8"))

# gather entries robustly
entries = []
if isinstance(data, list):
    entries = data
elif isinstance(data, dict):
    for v in data.values():
        if isinstance(v, list) and v and isinstance(v[0], dict) and "model_name" in v[0]:
            entries.extend(v)
    if "assistant_manual_evaluations" in data and isinstance(data["assistant_manual_evaluations"], list):
        entries.extend(data["assistant_manual_evaluations"])
    if not entries and "model_name" in data:
        entries = [data]

rows = []
for e in entries:
    if not isinstance(e, dict):
        continue
    name = e.get("model_name") or e.get("model")
    # detect manual score
    manual = None
    for k in ("assistant_manual_pedanticity", "manual_score", "human_score", "assistant_manual_score"):
        if k in e:
            manual = e.get(k); break
    # detect llm score
    llm = None
    for k in ("pedanticity_score", "llm_score", "score"):
        if k in e:
            llm = e.get(k); break
    rows.append({"model_name": name, "manual_score": manual, "llm_score": llm})

df = pd.DataFrame(rows).dropna(subset=["model_name"])
df["manual_score"] = pd.to_numeric(df["manual_score"], errors="coerce")
df["llm_score"] = pd.to_numeric(df["llm_score"], errors="coerce")
df["model_name"] = df["model_name"].astype(str)

if df.empty:
    raise SystemExit("No valid rows found.")

models = sorted(df["model_name"].unique())
n = len(models)
manual_by_model = [df.loc[df.model_name == m, "manual_score"].dropna().values for m in models]
llm_by_model = [df.loc[df.model_name == m, "llm_score"].dropna().values for m in models]

# replace empty arrays with nan array to avoid plotting errors
manual_by_model = [arr if arr.size else np.array([np.nan]) for arr in manual_by_model]
llm_by_model = [arr if arr.size else np.array([np.nan]) for arr in llm_by_model]

width = 0.35
indices = np.arange(n)
pos_m = indices - width / 2
pos_l = indices + width / 2

fig, ax = plt.subplots(figsize=(max(8, n * 1.2), 6))
bp_m = ax.boxplot(manual_by_model, positions=pos_m, widths=width * 0.9, patch_artist=True,
                  boxprops=dict(facecolor="#4C72B0"), medianprops=dict(color="black"), showfliers=False)
bp_l = ax.boxplot(llm_by_model, positions=pos_l, widths=width * 0.9, patch_artist=True,
                  boxprops=dict(facecolor="#DD8452"), medianprops=dict(color="black"), showfliers=False)

ax.set_xticks(indices)
ax.set_xticklabels(models, rotation=45, ha="right")
ax.set_xlim(-1, n)
ax.set_ylabel("Score (0–10)")
ax.set_title("Manual vs LLM evaluation by model")

# plot means
for i, (mvals, lvals) in enumerate(zip(manual_by_model, llm_by_model)):
    if not np.isnan(np.nanmean(mvals)):
        ax.plot(pos_m[i], np.nanmean(mvals), "o", color="black", markersize=4)
    if not np.isnan(np.nanmean(lvals)):
        ax.plot(pos_l[i], np.nanmean(lvals), "o", color="white", markeredgecolor="black", markersize=4)

ax.legend([bp_m["boxes"][0], bp_l["boxes"][0]], ["Manual", "LLM"], loc="upper right")
plt.tight_layout()
fig.savefig(OUT_PDF, format="pdf")
plt.close(fig)

agg = df.groupby("model_name").agg(
    manual_count=("manual_score", lambda s: int(s.notna().sum())),
    manual_mean=("manual_score", "mean"),
    manual_std=("manual_score", "std"),
    llm_count=("llm_score", lambda s: int(s.notna().sum())),
    llm_mean=("llm_score", "mean"),
    llm_std=("llm_score", "std"),
).reset_index()
agg.to_csv(OUT_CSV, index=False)

print(f"Saved: {OUT_PDF}")
print(f"Summary: {OUT_CSV}")


Saved: ../resources/model_boxplots.pdf
Summary: ../resources/model_summary.csv
